# 03 — Model Qiymətləndirmə və Vizuallaşdırma

Bu notebook train edilmiş YOLOv8n modelini test setində qiymətləndirir. Ultralytics-in avtomatik
hesabladığı metrikalar (`mAP@0.5`, `mAP@0.5:0.95`, `Precision`, `Recall`, `F1`) istifadə edilir.
Əlavə olaraq inference sürəti ölçülür, training loss curve, Precision-Recall curve və
test setindən predicted bounding box-lar vizuallaşdırılır.

## 1. İmportlar və Konfiqurasiya

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
from PIL import Image
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
import torch
device = 0 if torch.cuda.is_available() else 'cpu'
print('✅ Bütün importlar uğurla yükləndi.')
print(f'Device: {device}')

✅ Bütün importlar uğurla yükləndi.


In [19]:
# ── PATH KONFİQURASİYASI ──────────────────────────────────────────────────────
PROJECT_ROOT = Path(r'C:\Users\user\Desktop\project')
DATA_YAML    = PROJECT_ROOT / 'data.yaml'
RESULTS_DIR  = PROJECT_ROOT / 'runs' / 'exp1'
BEST_WEIGHTS = RESULTS_DIR / 'weights' / 'best.pt'

IMAGES_TEST  = PROJECT_ROOT / 'data' / 'images' / 'test'

CLASS_NAMES  = ['bicycle', 'bus', 'car', 'motorbike']

print(f'Model weights: {BEST_WEIGHTS}')
print(f'data.yaml    : {DATA_YAML}')
print(f'Test şəkilləri: {IMAGES_TEST}')
print(f'Weights mövcuddur: {BEST_WEIGHTS.exists()}')

Model weights: C:\Users\user\Desktop\project\runs\exp1\weights\best.pt
data.yaml    : C:\Users\user\Desktop\project\data.yaml
Test şəkilləri: C:\Users\user\Desktop\project\data\images\test
Weights mövcuddur: True


## 2. Modeli Yüklə

In [20]:
model = YOLO(str(BEST_WEIGHTS))
print(f'✅ best.pt yükləndi.')

✅ best.pt yükləndi.


## 3. Test Setində Qiymətləndirmə

In [ ]:
print('Test setində qiymətləndirmə aparılır...')

test_results = model.val(
    data   = str(DATA_YAML),
    split  = 'test',
    imgsz  = 640,
    batch  = 8,
    device = device,
    workers= 0,
    verbose= True,
    augment= True, 
    conf   = 0.20,
)

print('\n✅ Qiymətləndirmə tamamlandı.')

Test setində qiymətləndirmə aparılır...
Ultralytics 8.4.21  Python-3.11.14 torch-2.10.0+cpu CPU (11th Gen Intel Core(TM) i5-1135G7 2.40GHz)
val: Fast image access  (ping: 0.20.1 ms, read: 107.420.3 MB/s, size: 151.9 KB)
val: Scanning C:\Users\user\Desktop\project\data\labels\test.cache... 60 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1s/it 16.8s2.5s
                   all         60        129      0.803      0.577      0.681      0.497
               bicycle         19         19      0.875      0.737      0.812      0.646
                   bus         20         25      0.827        0.6      0.665      0.532
                   car         24         53      0.759      0.534      0.691      0.475
             motorbike         20         32      0.751      0.438      0.558      0.336
Speed: 2.4ms preprocess, 261.8ms inference, 0.0ms loss, 0.6ms postpr

In [40]:
# ── Ümumi metrikalar ──────────────────────────────────────────────────────────
map50    = test_results.box.map50
map5095  = test_results.box.map
mp       = test_results.box.mp      # mean precision
mr       = test_results.box.mr      # mean recall

print('=' * 50)
print('📊 TEST SET — ÜMUMI METRİKALAR')
print('=' * 50)
print(f'  mAP@0.5      : {map50:.4f}')
print(f'  mAP@0.5:0.95 : {map5095:.4f}')
print(f'  Precision    : {mp:.4f}')
print(f'  Recall       : {mr:.4f}')
print('=' * 50)

📊 TEST SET — ÜMUMI METRİKALAR
  mAP@0.5      : 0.6814
  mAP@0.5:0.95 : 0.4974
  Precision    : 0.8028
  Recall       : 0.5771


In [41]:
# ── Klass üzrə metrikalar ─────────────────────────────────────────────────────
print('\n📊 TEST SET — KLASS ÜZRƏ METRİKALAR')
header = f'{"Klass":<15} {"Precision":>10} {"Recall":>10} {"F1":>10} {"mAP@0.5":>10}'
print(header)
print('-' * len(header))

per_class_p    = test_results.box.p      # precision per class
per_class_r    = test_results.box.r      # recall per class
per_class_ap50 = test_results.box.ap50   # AP@0.5 per class

for i, cls in enumerate(CLASS_NAMES):
    p  = per_class_p[i]
    r  = per_class_r[i]
    f1 = 2 * p * r / (p + r + 1e-9)
    ap = per_class_ap50[i]
    print(f'{cls:<15} {p:>10.4f} {r:>10.4f} {f1:>10.4f} {ap:>10.4f}')

print('-' * len(header))
print(f'{"ORTALAMA":<15} {mp:>10.4f} {mr:>10.4f} {2*mp*mr/(mp+mr+1e-9):>10.4f} {map50:>10.4f}')


📊 TEST SET — KLASS ÜZRƏ METRİKALAR
Klass            Precision     Recall         F1    mAP@0.5
-----------------------------------------------------------
bicycle             0.8750     0.7367     0.7999     0.8117
bus                 0.8269     0.6000     0.6954     0.6649
car                 0.7588     0.5342     0.6270     0.6913
motorbike           0.7507     0.4375     0.5528     0.5575
-----------------------------------------------------------
ORTALAMA            0.8028     0.5771     0.6715     0.6814


## 4. Inference Sürəti

In [ ]:
def measure_inference_speed(model, images_dir: Path, n: int = 20, seed: int = 42) -> float:
    """
    Modelin inference sürətini ölçür.

    Args:
        model: Yüklənmiş YOLO modeli.
        images_dir: Test şəkillərinin qovluğu.
        n: Ölçüləcək şəkil sayı.
        seed: Random seed.

    Returns:
        Ortalama inference sürəti (ms/şəkil).
    """
    all_imgs = list(images_dir.glob('*.jpg'))
    random.seed(seed)
    samples = random.sample(all_imgs, min(n, len(all_imgs)))

    # Warmup
    model.predict(source=str(samples[0]), imgsz=640, device=device, verbose=False)
    times = []
    for img_path in samples:
        start = time.time()
        model.predict(source=str(img_path), imgsz=640, device=device, verbose=False)
        times.append((time.time() - start) * 1000)  # ms-ə çevir

    avg_ms = np.mean(times)
    return avg_ms


print(f'{20} şəkil üzərində inference sürəti ölçülür...')
avg_ms = measure_inference_speed(model, IMAGES_TEST, n=20, seed=SEED)
print(f'\n⚡ Ortalama inference sürəti: {avg_ms:.1f} ms/şəkil (CPU)')

20 şəkil üzərində inference sürəti ölçülür...

⚡ Ortalama inference sürəti: 90.7 ms/şəkil (CPU)


## 5. Training Loss Curve

In [43]:
results_png = RESULTS_DIR / 'results.png'
if results_png.exists():
    img = mpimg.imread(results_png)
    plt.figure(figsize=(18, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training və Validation Loss Curve', fontsize=14)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ training_curves.png saxlanıldı.')

<Figure size 1800x800 with 1 Axes>

✅ training_curves.png saxlanıldı.


## 6. Precision-Recall Curve

In [44]:
plt.close('all')
fig, ax = plt.subplots(1, 1, figsize=(9, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

# test_results.box.pr_data-dan götür
prec_data = test_results.box.p  # per class precision
rec_data  = test_results.box.r  # per class recall

for i, cls in enumerate(CLASS_NAMES):
    ap = test_results.box.ap50[i]
    ax.scatter(rec_data[i], prec_data[i],
               color=colors[i], s=100, zorder=5)
    ax.annotate(f'{cls} (AP={ap:.3f})',
                (rec_data[i], prec_data[i]),
                textcoords='offset points', xytext=(5, 5),
                color=colors[i], fontsize=10)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('Precision-Recall — Klass üzrə')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ pr_curve.png saxlanıldı.')

<Figure size 900x600 with 1 Axes>

✅ pr_curve.png saxlanıldı.


## 7. Test Setindən Predicted Bounding Box-lar

In [ ]:
def visualize_predictions(model, images_dir: Path, class_names: list,
                           n: int = 10, conf: float = 0.25, seed: int = 42) -> None:
    """
    Test şəkillərində modelin predicted bbox-larını göstərir.

    Args:
        model: Yüklənmiş YOLO modeli.
        images_dir: Test şəkillərinin qovluğu.
        class_names: Klass adlarının siyahısı.
        n: Göstəriləcək şəkil sayı.
        conf: Minimum confidence threshold.
        seed: Random seed.
    """
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
    all_imgs = list(images_dir.glob('*.jpg'))
    random.seed(seed)
    samples = random.sample(all_imgs, min(n, len(all_imgs)))

    cols = 3
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
    axes = axes.flatten()

    for i, img_path in enumerate(samples):
        results = model.predict(
            source=str(img_path), imgsz=640,
            conf=conf, device=device, verbose=False
        )[0]

        img = Image.open(img_path).convert('RGB')
        ax  = axes[i]
        ax.imshow(img)

        boxes = results.boxes
        if boxes is not None and len(boxes) > 0:
            for box in boxes:
                cls_id = int(box.cls.item())
                conf_val = box.conf.item()
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                color = colors[cls_id % len(colors)]
                rect = patches.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1 - 5,
                        f'{class_names[cls_id]} {conf_val:.2f}',
                        color=color, fontsize=9, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6))
        else:
            ax.text(0.5, 0.5, 'Heç bir obyekt tapılmadı',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=10, color='red')

        ax.set_title(f'{img_path.name}', fontsize=8)
        ax.axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Test Set — Predicted Bounding Box-lar', fontsize=14)
    plt.tight_layout()
    out_path = PROJECT_ROOT / 'predictions.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ predictions.png saxlanıldı: {out_path}')


visualize_predictions(model, IMAGES_TEST, CLASS_NAMES, n=10, conf=0.25, seed=SEED)

<Figure size 1800x2000 with 12 Axes>

✅ predictions.png saxlanıldı: C:\Users\user\Desktop\project\predictions.png


## 8. Xəta Analizi — Aşağı Confidence-lı Predictionlar

In [ ]:
def analyze_failures(model, images_dir: Path, class_names: list,
                      conf_threshold: float = 0.25, seed: int = 42) -> None:
    """
    Modelin heç bir obyekt detect edə bilmədiyi şəkilləri tapır və göstərir.

    Args:
        model: Yüklənmiş YOLO modeli.
        images_dir: Test şəkillərinin qovluğu.
        class_names: Klass adlarının siyahısı.
        conf_threshold: Minimum confidence threshold.
        seed: Random seed.
    """
    all_imgs = list(images_dir.glob('*.jpg'))
    failed = []

    for img_path in all_imgs:
        results = model.predict(
            source=str(img_path), imgsz=640,
            conf=conf_threshold, device=device, verbose=False
        )[0]
        if results.boxes is None or len(results.boxes) == 0:
            failed.append(img_path)

    print(f'Heç bir obyekt detect edilməyən şəkil sayı: {len(failed)} / {len(all_imgs)}')

    if not failed:
        print('✅ Bütün şəkillərdə ən azı 1 obyekt detect edildi.')
        return

    # İlk 6-sını göstər
    random.seed(seed)
    samples = random.sample(failed, min(6, len(failed)))
    cols = 3
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
    axes = axes.flatten()

    for i, img_path in enumerate(samples):
        axes[i].imshow(Image.open(img_path))
        axes[i].set_title(img_path.name, fontsize=8)
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Xəta Analizi — Detect Edilməyən Şəkillər', fontsize=13)
    plt.tight_layout()
    out_path = PROJECT_ROOT / 'failure_cases.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ failure_cases.png saxlanıldı: {out_path}')


analyze_failures(model, IMAGES_TEST, CLASS_NAMES, conf_threshold=0.25, seed=SEED)

Heç bir obyekt detect edilməyən şəkil sayı: 5 / 60


<Figure size 1500x800 with 6 Axes>

✅ failure_cases.png saxlanıldı: C:\Users\user\Desktop\project\failure_cases.png


## 9. Yekun Metrika Cədvəli

In [47]:
print('=' * 60)
print('📊 YEKun METRİKA CƏDVƏLİ — README üçün')
print('=' * 60)
print(f'  mAP@0.5          : {map50:.4f}')
print(f'  mAP@0.5:0.95     : {map5095:.4f}')
print(f'  Mean Precision   : {mp:.4f}')
print(f'  Mean Recall      : {mr:.4f}')
print(f'  Mean F1          : {2*mp*mr/(mp+mr+1e-9):.4f}')
print(f'  Inference sürəti : {avg_ms:.1f} ms/şəkil (CPU)')
print()
print('Klass üzrə:')
for i, cls in enumerate(CLASS_NAMES):
    p  = per_class_p[i]
    r  = per_class_r[i]
    f1 = 2 * p * r / (p + r + 1e-9)
    ap = per_class_ap50[i]
    print(f'  {cls:<12}: P={p:.3f}  R={r:.3f}  F1={f1:.3f}  AP50={ap:.3f}')
print('=' * 60)

📊 YEKun METRİKA CƏDVƏLİ — README üçün
  mAP@0.5          : 0.6814
  mAP@0.5:0.95     : 0.4974
  Mean Precision   : 0.8028
  Mean Recall      : 0.5771
  Mean F1          : 0.6715
  Inference sürəti : 90.7 ms/şəkil (CPU)

Klass üzrə:
  bicycle     : P=0.875  R=0.737  F1=0.800  AP50=0.812
  bus         : P=0.827  R=0.600  F1=0.695  AP50=0.665
  car         : P=0.759  R=0.534  F1=0.627  AP50=0.691
  motorbike   : P=0.751  R=0.438  F1=0.553  AP50=0.558
